In [0]:
print("RETAIL DATA ENGINEERING PROJECT")
print("=" * 50)

print("Spark Version:", spark.version)

print("\nAvailable catalogs:")
display(spark.sql("SHOW CATALOGS"))

print("\nAvailable databases/schemas:")
display(spark.sql("SHOW SCHEMAS"))

RETAIL DATA ENGINEERING PROJECT
Spark Version: 4.2.0

Available catalogs:


catalog
etl
etl1
samples
system
workspace



Available databases/schemas:


databaseName
default
information_schema


In [0]:
catalogs = spark.sql("SHOW CATALOGS").collect()

print("AVAILABLE CATALOGS:")
for row in catalogs:
    print(row[0])

print("\nCURRENT CATALOG:")
print(spark.sql("SELECT CURRENT_CATALOG()").first()[0])

print("\nCURRENT SCHEMA:")
print(spark.sql("SELECT CURRENT_SCHEMA()").first()[0])

AVAILABLE CATALOGS:
etl
etl1
samples
system
workspace

CURRENT CATALOG:
workspace

CURRENT SCHEMA:
default


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.retail_warehouse
""")

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail_warehouse")

print("Current catalog:", spark.sql("SELECT CURRENT_CATALOG()").first()[0])
print("Current schema:", spark.sql("SELECT CURRENT_SCHEMA()").first()[0])

display(spark.sql("SHOW TABLES"))

Current catalog: workspace
Current schema: retail_warehouse


database,tableName,isTemporary


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.retail_warehouse.raw_data;

DESCRIBE VOLUME workspace.retail_warehouse.raw_data;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_data,workspace,retail_warehouse,tanuja.cool123@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/workspace/retail_warehouse/raw_data/"

files = dbutils.fs.ls(raw_path)

for file in files:
    print(file.name, file.size, "bytes")

year_2009-2010.csv 47335960 bytes
year_2010-2011.csv 48581694 bytes


In [0]:
file_1 = spark.read.option("header", True).csv(
    raw_path + "year_2009-2010.csv"
)

file_2 = spark.read.option("header", True).csv(
    raw_path + "year_2010-2011.csv"
)

print("Year 2009-2010:", file_1.count())
print("Year 2010-2011:", file_2.count())

display(file_1.limit(5))

Year 2009-2010: 525461
Year 2010-2011: 541910


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [0]:
from pyspark.sql import functions as F

# 1. Source location
raw_path = "/Volumes/workspace/retail_warehouse/raw_data/"

# 2. Read CSV files
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(raw_path + "*.csv")
)

# 3. Standardize column names
for column in bronze_df.columns:
    new_column = (
        column.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    bronze_df = bronze_df.withColumnRenamed(column, new_column)

# 4. Add ingestion metadata
bronze_df = (
    bronze_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

# 5. Write Delta table
table_name = "workspace.retail_warehouse.bronze_retail_transactions"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

# 6. Validate
result = spark.table(table_name)

print("=" * 50)
print("BRONZE INGESTION SUCCESSFUL")
print("=" * 50)

print("Total records:", result.count())
print("Total columns:", len(result.columns))
print("Columns:", result.columns)

display(result.limit(10))

BRONZE INGESTION SUCCESSFUL
Total records: 1067371
Total columns: 10
Columns: ['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'ingestion_timestamp', 'source_file']


invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,ingestion_timestamp,source_file
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,2026-11-08T10:56:02.740Z,dbfs:/Volumes/workspace/retail_warehouse/raw_data/year_2009-2010.csv
